# SNC — Automated Training Sweep

Runs every variant listed in `experiments/experiments.json`, one
after another, fully unattended. Each variant is trained, diagnosed,
and evaluated (SI-SDRi + detection F1); a row is appended to
`experiments/results.csv` on Drive. Crashes are logged and the loop
moves on, so a single bad config doesn't waste the whole sweep.

Recommended runtime: Colab Pro+ background execution + T4 GPU. Open
this notebook, *Runtime → Run all*, close the tab. Come back when the
email notification fires and pick the winner via the webapp dropdown.

See `docs/automation_guide.md` for the full design.

## 1. Mount Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/snc'
GITHUB_USER = 'keremtutumlu'
GITHUB_REPO_NAME = 'selective-noise-cancelling'
BRANCH = 'feature/separator-quality-overhaul'

import os, subprocess, shutil
from pathlib import Path

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if token:
    clone_url = f'https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
    print('Using GITHUB_TOKEN from Colab Secrets.')
else:
    clone_url = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
    print('No GITHUB_TOKEN secret found — attempting anonymous clone.')

REPO_DIR = Path(f'/content/{GITHUB_REPO_NAME}')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
result = subprocess.run(
    ['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(
        f'git clone failed (exit {result.returncode}). '
        'If the repo is private, add a GITHUB_TOKEN secret in the left sidebar.'
    )
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-5'], check=True)

## 2. Symlink data, saved_models, and experiments to Drive

Models from every sweep accumulate on Drive so they survive container
restarts and are visible in the webapp dropdown.

In [ ]:
drive_data = Path(DRIVE_ROOT) / 'data'
drive_models = Path(DRIVE_ROOT) / 'saved_models'
drive_experiments = Path(DRIVE_ROOT) / 'experiments'
drive_data.mkdir(parents=True, exist_ok=True)
drive_models.mkdir(parents=True, exist_ok=True)
drive_experiments.mkdir(parents=True, exist_ok=True)

# Copy the in-repo experiments.json to Drive on first run so you can
# edit it from a phone between sweeps without re-cloning the repo.
repo_cfg = REPO_DIR / 'experiments' / 'experiments.json'
drive_cfg = drive_experiments / 'experiments.json'
if not drive_cfg.exists() and repo_cfg.exists():
    shutil.copy(repo_cfg, drive_cfg)
    print(f'Seeded {drive_cfg} from repo.')

for local, target in [(REPO_DIR / 'data', drive_data),
                       (REPO_DIR / 'saved_models', drive_models),
                       (REPO_DIR / 'experiments', drive_experiments)]:
    if local.is_symlink() or local.exists():
        local.unlink() if local.is_symlink() else shutil.rmtree(local)
    local.symlink_to(target)
    print(f'{local} -> {target}')

## Stage 0a — Download ESC-50 (only if missing)

In [ ]:
import zipfile, urllib.request

archive_dir = REPO_DIR / 'data' / 'raw' / 'archive'
csv_path = archive_dir / 'esc50.csv'
wav_dir = archive_dir / 'audio' / 'audio'

if csv_path.exists() and wav_dir.exists() and len(list(wav_dir.glob('*.wav'))) >= 2000:
    print(f'ESC-50 already present — {len(list(wav_dir.glob("*.wav")))} WAV files.')
else:
    archive_dir.mkdir(parents=True, exist_ok=True)
    zip_path = Path('/content/esc50.zip')
    url = 'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip'
    print('Downloading ESC-50 (~600 MB on Colab network)...')
    urllib.request.urlretrieve(url, zip_path)

    extract_dir = Path('/content/esc50_extracted')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    src_root = extract_dir / 'ESC-50-master'

    shutil.copy(src_root / 'meta' / 'esc50.csv', csv_path)
    wav_dir.mkdir(parents=True, exist_ok=True)
    print('Copying WAV files to Drive...')
    for wav in (src_root / 'audio').glob('*.wav'):
        shutil.copy(wav, wav_dir / wav.name)

    zip_path.unlink()
    shutil.rmtree(extract_dir)
    print(f'Done — {len(list(wav_dir.glob("*.wav")))} WAV files.')

assert csv_path.exists() and wav_dir.exists()

In [ ]:
# Stage 0b — Download UrbanSound8K (~6 GB, only if missing).
# Adds 10 urban classes; 4 merge into ESC-50 classes, 6 are new.
import tarfile

us8k_dir = REPO_DIR / 'data' / 'raw' / 'urbansound8k'
us8k_csv = us8k_dir / 'metadata' / 'UrbanSound8K.csv'

if us8k_csv.exists():
    print('UrbanSound8K already present.')
else:
    tar_path = Path('/content/urbansound8k.tar.gz')
    url = 'https://zenodo.org/record/1203745/files/UrbanSound8K.tar.gz'
    print('Downloading UrbanSound8K (~6 GB on Colab network)...')
    urllib.request.urlretrieve(url, tar_path)

    print('Extracting...')
    extract_dir = Path('/content/us8k_extract')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    with tarfile.open(tar_path) as t:
        t.extractall(extract_dir)

    us8k_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(extract_dir / 'UrbanSound8K'), str(us8k_dir))
    tar_path.unlink()
    shutil.rmtree(extract_dir, ignore_errors=True)
    print('UrbanSound8K ready at', us8k_dir)

assert us8k_csv.exists()

In [ ]:
# Stage 0c — Download FSD50K (~34 GB total, only if missing).
# Adds ~200 AudioSet-derived classes (~51 k clips) on top of ESC-50 + US8K.
# Each piece is checked independently so a partial previous run (e.g. CSVs
# downloaded but audio download timed out) resumes from where it stopped.
import time

fsd_dir = REPO_DIR / 'data' / 'raw' / 'fsd50k'
fsd_dev_audio = fsd_dir / 'FSD50K.dev_audio'
fsd_eval_audio = fsd_dir / 'FSD50K.eval_audio'
fsd_dev_csv = fsd_dir / 'FSD50K.ground_truth' / 'dev.csv'
fsd_eval_csv = fsd_dir / 'FSD50K.ground_truth' / 'eval.csv'

fsd_dir.mkdir(parents=True, exist_ok=True)
base = 'https://zenodo.org/record/4060432/files'


def _fetch(name, max_retries=3):
    out = fsd_dir / name
    if out.exists() and out.stat().st_size > 0:
        print(f'  {name} already downloaded ({out.stat().st_size / 1e9:.2f} GB).')
        return out
    for attempt in range(1, max_retries + 1):
        try:
            print(f'  Downloading {name} (attempt {attempt})...', end=' ', flush=True)
            urllib.request.urlretrieve(f'{base}/{name}', out)
            print(f'{out.stat().st_size / 1e9:.2f} GB.')
            return out
        except Exception as exc:
            print(f'failed: {exc}')
            if out.exists():
                out.unlink()
            if attempt == max_retries:
                raise
            time.sleep(2 ** attempt)
    return out


def _has_audio(audio_dir):
    return audio_dir.is_dir() and any(audio_dir.glob('*.wav'))


def _download_split(part_names, combined_name, target_dir):
    for part in part_names:
        _fetch(part)
    final = fsd_dir / part_names[-1]
    combined = fsd_dir / combined_name
    print(f'  Combining {len(part_names)} split parts into {combined_name}...')
    subprocess.run(['zip', '-FF', str(final),
                    '--out', str(combined), '-q'], check=True)
    print(f'  Extracting {combined_name}...')
    with zipfile.ZipFile(combined) as z:
        z.extractall(fsd_dir)
    for p in part_names:
        (fsd_dir / p).unlink(missing_ok=True)
    combined.unlink(missing_ok=True)
    if not _has_audio(target_dir):
        raise RuntimeError(f'Extraction finished but {target_dir} has no .wav files.')


if fsd_dev_csv.exists() and fsd_eval_csv.exists():
    print('FSD50K ground-truth CSVs already present.')
else:
    print('Fetching FSD50K ground truth...')
    gt = _fetch('FSD50K.ground_truth.zip')
    with zipfile.ZipFile(gt) as z:
        z.extractall(fsd_dir)
    gt.unlink()

if _has_audio(fsd_dev_audio):
    print(f'FSD50K dev audio already present '
          f'({sum(1 for _ in fsd_dev_audio.glob("*.wav"))} .wav files).')
else:
    print('Fetching FSD50K dev audio (6-part archive, expect ~45-90 min)...')
    _download_split(
        ['FSD50K.dev_audio.z01', 'FSD50K.dev_audio.z02',
         'FSD50K.dev_audio.z03', 'FSD50K.dev_audio.z04',
         'FSD50K.dev_audio.z05', 'FSD50K.dev_audio.zip'],
        '_dev_combined.zip', fsd_dev_audio,
    )

if _has_audio(fsd_eval_audio):
    print(f'FSD50K eval audio already present '
          f'({sum(1 for _ in fsd_eval_audio.glob("*.wav"))} .wav files).')
else:
    print('Fetching FSD50K eval audio (2-part archive)...')
    _download_split(
        ['FSD50K.eval_audio.z01', 'FSD50K.eval_audio.zip'],
        '_eval_combined.zip', fsd_eval_audio,
    )

assert fsd_dev_csv.exists() and _has_audio(fsd_dev_audio) and _has_audio(fsd_eval_audio)
print(f'FSD50K ready at {fsd_dir}')

## 3. Install dependencies

In [ ]:
!pip install -q librosa==0.11.0 soundfile scikit-learn pandas

## 4. Load the experiment list

Each entry's `id` becomes the model filename suffix
(`separator_unet_film_multi_<id>.h5`). All other fields override
`ConditionedSeparatorTrainer` defaults — anything its `__init__` accepts
(including mixer params like `negative_prob`, `bg_noise_prob`,
`bg_snr_db_range`, `max_mix`) can be set here.

In [ ]:
import json

EXPERIMENTS_CFG = REPO_DIR / 'experiments' / 'experiments.json'
with EXPERIMENTS_CFG.open() as f:
    experiments = json.load(f)['experiments']
print(f'Loaded {len(experiments)} experiment(s):')
for e in experiments:
    overrides = ', '.join(f'{k}={v}' for k, v in e.items() if k != 'id')
    print(f'  - {e["id"]}  ({overrides or "defaults"})')

## 5. Run the sweep

Each experiment: train → diagnose → SI-SDRi → detection F1, then
append a row to `experiments/results.csv`. Crashes are caught and
logged so the loop continues to the next variant.

In [ ]:
import csv, gc, sys, traceback
from datetime import datetime

import tensorflow as tf

sys.path.insert(0, str(REPO_DIR / 'src' / 'data_preparation'))
sys.path.insert(0, str(REPO_DIR / 'src' / 'model_training'))

MODEL_DIR = REPO_DIR / 'saved_models' / 'separation_models'
DATA_ROOT = REPO_DIR / 'data' / 'raw'
RESULTS_CSV = REPO_DIR / 'experiments' / 'results.csv'

RESULT_COLUMNS = [
    'id', 'timestamp', 'status',
    'negative_prob', 'bg_noise_prob', 'epochs', 'max_mix', 'base_filters',
    'diagnose_pass', 'diagnose_mean_ratio',
    'si_sdri', 'si_sdr_model', 'si_sdr_mix',
    'f1', 'tp', 'fp', 'fn',
    'error',
]


def _append_row(row):
    write_header = not RESULTS_CSV.exists()
    with RESULTS_CSV.open('a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=RESULT_COLUMNS)
        if write_header:
            w.writeheader()
        w.writerow({k: row.get(k, '') for k in RESULT_COLUMNS})


def _run_one(cfg):
    """Train + evaluate one experiment. Returns a result row."""
    exp_id = cfg['id']
    overrides = {k: v for k, v in cfg.items() if k != 'id'}
    if 'bg_snr_db_range' in overrides:
        overrides['bg_snr_db_range'] = tuple(overrides['bg_snr_db_range'])

    row = {
        'id': exp_id,
        'timestamp': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
        'status': 'running',
        'negative_prob': overrides.get('negative_prob', 0.15),
        'bg_noise_prob': overrides.get('bg_noise_prob', 0.10),
        'epochs': overrides.get('epochs', 60),
        'max_mix': overrides.get('max_mix', 4),
        'base_filters': overrides.get('base_filters', 32),
    }

    # Each run starts with a fresh TF session so previous models don't pin
    # GPU memory and weight-name collisions can't happen.
    tf.keras.backend.clear_session()
    gc.collect()

    from train_conditioned_separator import ConditionedSeparatorTrainer
    trainer = ConditionedSeparatorTrainer(
        data_root=DATA_ROOT,
        model_save_dir=MODEL_DIR,
        model_version=exp_id,
        **overrides,
    )
    print(f'\n{"=" * 70}\n>>> Training {exp_id}\n{"=" * 70}\n')
    trainer.train()
    model_path = trainer.checkpoint_path

    # Diagnose
    tf.keras.backend.clear_session(); gc.collect()
    from diagnose_model import diagnose
    diag = diagnose(model_path=model_path, data_root=DATA_ROOT)
    row['diagnose_pass'] = diag['pass']
    row['diagnose_mean_ratio'] = f"{diag['mean_out_in_ratio']:.4f}"

    # SI-SDRi
    tf.keras.backend.clear_session(); gc.collect()
    from evaluate_conditioned_separator import ConditionedSeparatorEvaluator
    sdr = ConditionedSeparatorEvaluator(
        data_root=DATA_ROOT, model_path=model_path).evaluate()
    row['si_sdri'] = f"{sdr['si_sdri']:.2f}"
    row['si_sdr_model'] = f"{sdr['si_sdr_model']:.2f}"
    row['si_sdr_mix'] = f"{sdr['si_sdr_mix']:.2f}"

    # Detection F1
    tf.keras.backend.clear_session(); gc.collect()
    from evaluate_detection import evaluate as eval_detection
    det = eval_detection(model_path=model_path, data_root=DATA_ROOT)
    row['f1'] = f"{det['f1_mean']:.3f}"
    row['tp'] = det['tp_total']
    row['fp'] = det['fp_total']
    row['fn'] = det['fn_total']

    row['status'] = 'ok'
    return row


for cfg in experiments:
    try:
        row = _run_one(cfg)
    except Exception as exc:
        traceback.print_exc()
        row = {
            'id': cfg.get('id', '<unknown>'),
            'timestamp': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
            'status': 'crashed',
            'error': f'{type(exc).__name__}: {exc}'[:200],
        }
    _append_row(row)
    print(f'\n>>> Logged result for {row["id"]} (status={row["status"]})')

## 6. Leaderboard

All runs so far, sorted by SI-SDRi descending. Open this CSV from any
machine (it lives on Drive) to track progress mid-sweep.

In [ ]:
import pandas as pd

df = pd.read_csv(RESULTS_CSV)
df_sorted = df.sort_values('si_sdri', ascending=False, na_position='last')
cols = ['id', 'status', 'negative_prob', 'diagnose_pass',
        'si_sdri', 'f1', 'tp', 'fp', 'fn', 'timestamp']
print(df_sorted[cols].to_string(index=False))

# Also write a markdown snapshot for easy phone viewing.
md_path = REPO_DIR / 'experiments' / 'results.md'
with md_path.open('w') as f:
    f.write('# Sweep leaderboard\n\n')
    f.write(df_sorted[cols].to_markdown(index=False))
    f.write('\n')
print(f'\nLeaderboard also written to {md_path}')